# ATR Volatility Breakout v2 — High Frequency Edition

**Problem with v1:** Only 38 trades over 8 years (4.6/yr) on NVDA. Sharpe estimates unreliable with <50 trades. Returns dominated by a few monster wins.

**What we changed:**

| Feature | v1 | v2 |
|---|---|---|
| ATR Multiplier | 1.5 (conservative) | 0.75–1.25 sweep (more entries) |
| Exit | Hard signal reversal | Trailing ATR stop (ride trends, re-enter) |
| Re-entry | Wait for full new breakout | Pullback re-entry within uptrend |
| Confirmation | None | ADX > 20 + Volume spike filter |
| Assets | Single (NVDA) | Multi-asset rotation (6-8 tickers) |
| Target trades | ~5/yr/asset | 15-30/yr/asset |

**Goal:** 3-5x more trades without destroying the edge. Better statistical reliability and smoother FTMO Monte Carlo.

---

## 1. Environment & Dependencies

In [ ]:
# !pip install yfinance TA-Lib vectorbt scipy pandas numpy matplotlib --quiet

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 140)

print("All imports loaded")
print(f"   vectorbt: {vbt.__version__}  |  pandas: {pd.__version__}  |  numpy: {np.__version__}")

## 2. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — ATR Volatility Breakout v2
# ═══════════════════════════════════════════════════════════════

# Multi-asset universe
TICKERS = ["NVDA", "AMD", "MSFT", "AMZN", "META", "GOOGL", "AAPL", "QQQ"]
START_DATE    = "2018-01-01"
END_DATE      = None
TRAIN_RATIO   = 0.60
ANNUAL_FACTOR = 252          # Equities

# ─── ATR Breakout Parameters (v1 baseline for comparison) ───
V1_ATR_PERIOD = 25
V1_SMA_PERIOD = 10
V1_ATR_MULT   = 1.5

# ─── v2 Parameter Grid (small — just threshold sweep) ───
ATR_PERIODS   = [10, 14, 20]
SMA_PERIODS   = [5, 10]
ATR_MULTS     = [0.75, 1.0, 1.25]

# ─── Trailing Stop ───
TRAIL_ATR_MULT = 2.0         # Exit when price drops 2x ATR from highest close

# ─── Confirmation Filters ───
ADX_LEN       = 14
ADX_MIN       = 20           # Only enter when trend is present
VOL_SPIKE_MULT = 1.3         # Volume must be 1.3x its 20-day average

# ─── Pullback Re-entry ───
PULLBACK_ENABLED = True
PULLBACK_ATR_MULT = 0.5      # Re-enter when price pulls back 0.5x ATR to SMA
REGIME_SMA_LEN = 50          # Must be above 50-SMA for pullback re-entry

# ─── Execution ───
SIGNAL_SHIFT  = 1
FEES          = 0.0005
SLIPPAGE      = 0.0005
INIT_CASH     = 100_000
MIN_TRADES    = 10

# ─── FTMO ───
FTMO_ACCOUNT       = 100_000
FTMO_PROFIT_TARGET = 0.10
FTMO_MAX_DAILY_DD  = 0.05
FTMO_MAX_TOTAL_DD  = 0.10
FTMO_CHALLENGE_DAYS = 30
MONTE_CARLO_PATHS  = 10_000

total_combos = len(ATR_PERIODS) * len(SMA_PERIODS) * len(ATR_MULTS)
print(f"Configuration loaded")
print(f"   Tickers: {TICKERS}")
print(f"   v2 grid: {total_combos} combos (ATR periods x SMA periods x ATR mults)")
print(f"   Trailing stop: {TRAIL_ATR_MULT}x ATR")
print(f"   Confirmation: ADX > {ADX_MIN}, Volume > {VOL_SPIKE_MULT}x avg")
print(f"   Pullback re-entry: {'ON' if PULLBACK_ENABLED else 'OFF'}")

## 3. Data Download — Multi-Asset

In [ ]:
# Download all tickers
all_data = {}
for ticker in TICKERS:
    raw = yf.download(ticker, start=START_DATE, end=END_DATE)
    if isinstance(raw.columns, pd.MultiIndex):
        raw = raw.droplevel('Ticker', axis=1)
    raw = raw.ffill().bfill()
    
    all_data[ticker] = {
        'close': raw['Close'].astype(float),
        'high': raw['High'].astype(float),
        'low': raw['Low'].astype(float),
        'volume': raw['Volume'].astype(float),
        'df': raw,
    }
    bars = len(raw)
    print(f"   {ticker}: {bars} bars | {raw.index[0].date()} -> {raw.index[-1].date()}")

print(f"\nLoaded {len(all_data)} assets")

## 4. v2 Signal Engine — Breakout + Trailing Stop + Pullback Re-entry

The core logic:
1. **Breakout entry**: Close > SMA + ATR * mult, confirmed by ADX and volume
2. **Trailing ATR stop**: Track highest close since entry, exit when price drops 2x ATR below it
3. **Pullback re-entry**: If still in uptrend (above regime SMA), re-enter when price bounces off SMA after a pullback

In [ ]:
def generate_v2_signals(close, high, low, volume,
                         atr_period, sma_period, atr_mult,
                         trail_atr_mult=TRAIL_ATR_MULT,
                         adx_min=ADX_MIN, vol_spike_mult=VOL_SPIKE_MULT,
                         pullback=PULLBACK_ENABLED,
                         pullback_atr_mult=PULLBACK_ATR_MULT,
                         regime_sma_len=REGIME_SMA_LEN):
    """
    ATR Volatility Breakout v2 with trailing stop and pullback re-entry.
    Returns (entries, exits) as boolean Series.
    
    Entry conditions (breakout):
      - Close > SMA + ATR * mult
      - ADX > threshold (trend present)
      - Volume > vol_spike_mult * 20-day avg volume
    
    Entry conditions (pullback re-entry):
      - Close > regime SMA (still in uptrend)
      - Close touched or crossed below SMA (pulled back)
      - Close crosses back above SMA + ATR * pullback_mult
      - ADX > threshold
    
    Exit: Trailing stop — price drops trail_atr_mult * ATR below highest close since entry
    """
    n = len(close)
    idx = close.index
    c = close.values.astype(float)
    h = high.values.astype(float)
    l = low.values.astype(float)
    v = volume.values.astype(float)
    
    # Indicators
    atr = talib.ATR(h, l, c, timeperiod=atr_period)
    sma = talib.SMA(c, timeperiod=sma_period)
    adx = talib.ADX(h, l, c, timeperiod=ADX_LEN)
    vol_avg = talib.SMA(v, timeperiod=20)
    regime_sma = talib.SMA(c, timeperiod=regime_sma_len)
    
    entries = np.zeros(n, dtype=bool)
    exits = np.zeros(n, dtype=bool)
    
    in_trade = False
    highest_since_entry = 0.0
    touched_sma = False  # For pullback tracking
    
    for i in range(max(sma_period, atr_period, regime_sma_len, 20), n):
        if np.isnan(atr[i]) or np.isnan(sma[i]) or np.isnan(adx[i]) or np.isnan(vol_avg[i]):
            continue
            
        if in_trade:
            # Update trailing high
            if c[i] > highest_since_entry:
                highest_since_entry = c[i]
            
            # Trailing stop check
            trail_stop = highest_since_entry - trail_atr_mult * atr[i]
            if c[i] < trail_stop:
                exits[i] = True
                in_trade = False
                touched_sma = False
        else:
            # ── Breakout entry ──
            breakout_level = sma[i] + atr_mult * atr[i]
            adx_ok = adx[i] > adx_min
            vol_ok = v[i] > vol_spike_mult * vol_avg[i] if vol_avg[i] > 0 else False
            
            if c[i] > breakout_level and adx_ok and vol_ok:
                entries[i] = True
                in_trade = True
                highest_since_entry = c[i]
                touched_sma = False
                continue
            
            # ── Pullback re-entry ──
            if pullback and not np.isnan(regime_sma[i]):
                # Track if price pulled back to SMA
                if c[i] <= sma[i] + 0.1 * atr[i]:  # Close enough to SMA
                    touched_sma = True
                
                if touched_sma:
                    pullback_level = sma[i] + pullback_atr_mult * atr[i]
                    in_uptrend = c[i] > regime_sma[i]
                    
                    if c[i] > pullback_level and adx_ok and in_uptrend:
                        entries[i] = True
                        in_trade = True
                        highest_since_entry = c[i]
                        touched_sma = False
    
    return pd.Series(entries, index=idx), pd.Series(exits, index=idx)


def generate_v1_signals(close, high, low, atr_period, sma_period, atr_mult):
    """Original v1 signal logic for comparison."""
    c = close.values.astype(float)
    h = high.values.astype(float)
    l = low.values.astype(float)
    
    atr = pd.Series(talib.ATR(h, l, c, timeperiod=atr_period), index=close.index)
    sma = pd.Series(talib.SMA(c, timeperiod=sma_period), index=close.index)
    
    upper = sma + atr_mult * atr
    lower = sma - atr_mult * atr
    
    entries = (close > upper) & (close.shift(1) <= upper.shift(1))
    exits = (close < lower) & (close.shift(1) >= lower.shift(1))
    
    return entries.fillna(False), exits.fillna(False)


# Quick sanity check on NVDA
d = all_data['NVDA']
e_v2, x_v2 = generate_v2_signals(d['close'], d['high'], d['low'], d['volume'],
                                   atr_period=14, sma_period=10, atr_mult=1.0)
e_v1, x_v1 = generate_v1_signals(d['close'], d['high'], d['low'],
                                   V1_ATR_PERIOD, V1_SMA_PERIOD, V1_ATR_MULT)

print(f"Signal generators defined\n")
print(f"NVDA sanity check:")
print(f"   v1 (original):  {e_v1.sum()} entries, {x_v1.sum()} exits")
print(f"   v2 (new):       {e_v2.sum()} entries, {x_v2.sum()} exits")
print(f"   Ratio:          {e_v2.sum() / max(e_v1.sum(), 1):.1f}x more entries")

## 5. Train/Test Split & Backtest Helper

In [ ]:
def run_backtest(close, entries, exits, shift=SIGNAL_SHIFT):
    """Run vectorbt backtest with anti-lookahead shift. Returns Portfolio or None."""
    ent = entries.shift(shift).fillna(False).astype(bool)
    exi = exits.shift(shift).fillna(False).astype(bool)
    if ent.sum() < 3:
        return None
    return vbt.Portfolio.from_signals(close, entries=ent, exits=exi,
                                       init_cash=INIT_CASH, fees=FEES,
                                       slippage=SLIPPAGE, freq='D')

def split_data(data_dict, train_ratio=TRAIN_RATIO):
    """Split a single asset's data dict into train/test."""
    close = data_dict['close']
    split_idx = int(len(close) * train_ratio)
    train = {k: v.iloc[:split_idx] if hasattr(v, 'iloc') else v for k, v in data_dict.items()}
    test = {k: v.iloc[split_idx:] if hasattr(v, 'iloc') else v for k, v in data_dict.items()}
    return train, test, split_idx

def extract_metrics(pf, label=""):
    """Extract key metrics from a portfolio."""
    if pf is None:
        return {'label': label, 'sharpe': np.nan, 'sortino': np.nan, 'return': np.nan,
                'max_dd': np.nan, 'trades': 0, 'win_rate': np.nan, 'pf': np.nan, 'trades_yr': np.nan}
    trades = pf.trades.count()
    days = (pf.value().index[-1] - pf.value().index[0]).days
    years = max(days / 365.25, 0.01)
    return {
        'label': label,
        'sharpe': float(pf.sharpe_ratio()),
        'sortino': float(pf.sortino_ratio()),
        'return': float(pf.total_return()),
        'max_dd': float(pf.max_drawdown()),
        'trades': int(trades),
        'win_rate': float(pf.trades.win_rate()) if trades > 0 else np.nan,
        'pf': float(pf.trades.profit_factor()) if trades > 0 else np.nan,
        'trades_yr': trades / years,
    }

print("Backtest helpers defined")

## 6. v1 vs v2 Head-to-Head on NVDA (Training Data)

Run both versions on NVDA training data to see the trade count improvement before touching OOS.

In [ ]:
# Split NVDA
nvda = all_data['NVDA']
nvda_train, nvda_test, nvda_split = split_data(nvda)

print(f"NVDA split at index {nvda_split}")
print(f"   Train: {len(nvda_train['close'])} bars | {nvda_train['close'].index[0].date()} -> {nvda_train['close'].index[-1].date()}")
print(f"   Test:  {len(nvda_test['close'])} bars  | {nvda_test['close'].index[0].date()} -> {nvda_test['close'].index[-1].date()}")

# v1 baseline on training data
e_v1, x_v1 = generate_v1_signals(nvda_train['close'], nvda_train['high'], nvda_train['low'],
                                   V1_ATR_PERIOD, V1_SMA_PERIOD, V1_ATR_MULT)
pf_v1_train = run_backtest(nvda_train['close'], e_v1, x_v1)
m_v1 = extract_metrics(pf_v1_train, "v1 Original")

# v2 grid search on training data — small grid
print(f"\nGrid search v2 on NVDA training data ({total_combos} combos)...\n")
v2_results = []

for atr_p in ATR_PERIODS:
    for sma_p in SMA_PERIODS:
        for atr_m in ATR_MULTS:
            try:
                e, x = generate_v2_signals(nvda_train['close'], nvda_train['high'],
                                            nvda_train['low'], nvda_train['volume'],
                                            atr_period=atr_p, sma_period=sma_p, atr_mult=atr_m)
                pf = run_backtest(nvda_train['close'], e, x)
                if pf is None:
                    continue
                m = extract_metrics(pf, f"atr={atr_p},sma={sma_p},mult={atr_m}")
                m['atr_period'] = atr_p
                m['sma_period'] = sma_p
                m['atr_mult'] = atr_m
                v2_results.append(m)
            except:
                pass

v2_df = pd.DataFrame(v2_results).sort_values('sharpe', ascending=False)

# Print comparison
print(f"{'Config':<30} {'Sharpe':>8} {'Return':>10} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'WinRate':>8} {'PF':>6}")
print("-" * 95)
print(f"{'v1 (25/10/1.5)':<30} {m_v1['sharpe']:>8.3f} {m_v1['return']:>10.2%} {m_v1['max_dd']:>8.2%} {m_v1['trades']:>8} {m_v1['trades_yr']:>8.1f} {m_v1['win_rate']:>8.1%} {m_v1['pf']:>6.2f}")
print("-" * 95)
for _, r in v2_df.head(10).iterrows():
    label = f"v2 ({int(r['atr_period'])}/{int(r['sma_period'])}/{r['atr_mult']:.2f})"
    print(f"{label:<30} {r['sharpe']:>8.3f} {r['return']:>10.2%} {r['max_dd']:>8.2%} {int(r['trades']):>8} {r['trades_yr']:>8.1f} {r['win_rate']:>8.1%} {r['pf']:>6.2f}")

# Best v2
best_v2 = v2_df.iloc[0]
BEST_ATR_P = int(best_v2['atr_period'])
BEST_SMA_P = int(best_v2['sma_period'])
BEST_ATR_M = best_v2['atr_mult']

print(f"\nBest v2: atr_period={BEST_ATR_P}, sma_period={BEST_SMA_P}, atr_mult={BEST_ATR_M}")
print(f"   Trade count improvement: {m_v1['trades']} -> {int(best_v2['trades'])} ({int(best_v2['trades'])/max(m_v1['trades'],1):.1f}x)")

## 7. OOS Validation — NVDA v1 vs v2

In [ ]:
# v1 OOS
e_v1_oos, x_v1_oos = generate_v1_signals(nvda_test['close'], nvda_test['high'], nvda_test['low'],
                                           V1_ATR_PERIOD, V1_SMA_PERIOD, V1_ATR_MULT)
pf_v1_oos = run_backtest(nvda_test['close'], e_v1_oos, x_v1_oos)
m_v1_oos = extract_metrics(pf_v1_oos, "v1 OOS")

# v2 OOS (best params from training)
e_v2_oos, x_v2_oos = generate_v2_signals(nvda_test['close'], nvda_test['high'],
                                           nvda_test['low'], nvda_test['volume'],
                                           atr_period=BEST_ATR_P, sma_period=BEST_SMA_P,
                                           atr_mult=BEST_ATR_M)
pf_v2_oos = run_backtest(nvda_test['close'], e_v2_oos, x_v2_oos)
m_v2_oos = extract_metrics(pf_v2_oos, "v2 OOS")

# Buy & Hold OOS
pf_bh_oos = vbt.Portfolio.from_holding(nvda_test['close'], init_cash=INIT_CASH, fees=FEES, freq='D')
m_bh = extract_metrics(pf_bh_oos, "Buy & Hold")

# Comparison table
print(f"NVDA Out-of-Sample: v1 vs v2 vs Buy & Hold")
print(f"{'='*90}")
print(f"{'Metric':<18} {'v1 (original)':>15} {'v2 (new)':>15} {'Buy & Hold':>15}")
print(f"{'-'*90}")
for metric, fmt_fn in [('sharpe', lambda v: f"{v:.3f}"), ('sortino', lambda v: f"{v:.3f}"),
                         ('return', lambda v: f"{v:.2%}"), ('max_dd', lambda v: f"{v:.2%}"),
                         ('trades', lambda v: f"{int(v)}"), ('trades_yr', lambda v: f"{v:.1f}"),
                         ('win_rate', lambda v: f"{v:.1%}"), ('pf', lambda v: f"{v:.2f}")]:
    v1_val = m_v1_oos.get(metric, np.nan)
    v2_val = m_v2_oos.get(metric, np.nan)
    bh_val = m_bh.get(metric, np.nan)
    v1_str = fmt_fn(v1_val) if not np.isnan(v1_val) else "N/A"
    v2_str = fmt_fn(v2_val) if not np.isnan(v2_val) else "N/A"
    bh_str = fmt_fn(bh_val) if not np.isnan(bh_val) else "N/A"
    print(f"{metric:<18} {v1_str:>15} {v2_str:>15} {bh_str:>15}")

# Equity curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('NVDA OOS — v1 vs v2 Equity Curves', fontsize=14, fontweight='bold')

if pf_v1_oos:
    pf_v1_oos.value().plot(ax=axes[0], color='navy', linewidth=1.5, label=f'v1 (Sharpe: {m_v1_oos["sharpe"]:.3f})')
if pf_v2_oos:
    pf_v2_oos.value().plot(ax=axes[0], color='darkorange', linewidth=1.5, label=f'v2 (Sharpe: {m_v2_oos["sharpe"]:.3f})')
pf_bh_oos.value().plot(ax=axes[0], color='gray', linewidth=1, linestyle='--', alpha=0.6, label='Buy & Hold')
axes[0].set_title('Portfolio Value')
axes[0].set_ylabel('$')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Trade count comparison bar chart
labels = ['v1 Original', 'v2 New']
is_trades = [m_v1['trades'], int(best_v2['trades'])]
oos_trades = [m_v1_oos['trades'], m_v2_oos['trades']]
x = np.arange(len(labels))
axes[1].bar(x - 0.2, is_trades, 0.35, label='IS Trades', color='steelblue')
axes[1].bar(x + 0.2, oos_trades, 0.35, label='OOS Trades', color='darkorange')
axes[1].set_ylabel('Trade Count')
axes[1].set_title('Trade Count: v1 vs v2')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')
for i, (is_t, oos_t) in enumerate(zip(is_trades, oos_trades)):
    axes[1].text(i - 0.2, is_t + 0.5, str(is_t), ha='center', fontsize=10, fontweight='bold')
    axes[1].text(i + 0.2, oos_t + 0.5, str(oos_t), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Multi-Asset Scan — Best v2 Params Across All Tickers

Run the best v2 parameters on all 8 assets. This tells us if the strategy generalizes or if it's NVDA-specific.

In [ ]:
print(f"Multi-Asset Scan — v2 params: atr={BEST_ATR_P}, sma={BEST_SMA_P}, mult={BEST_ATR_M}")
print(f"{'='*110}")
print(f"{'Ticker':<8} {'IS Sharpe':>10} {'OOS Sharpe':>11} {'IS Trades':>10} {'OOS Trades':>11} "
      f"{'OOS Return':>11} {'OOS MaxDD':>10} {'OOS WR':>8} {'OOS PF':>8} {'Degrad%':>8}")
print(f"{'-'*110}")

multi_results = []
multi_portfolios = {}

for ticker in TICKERS:
    d = all_data[ticker]
    train, test, sidx = split_data(d)
    
    # IS
    try:
        e_is, x_is = generate_v2_signals(train['close'], train['high'], train['low'], train['volume'],
                                          atr_period=BEST_ATR_P, sma_period=BEST_SMA_P, atr_mult=BEST_ATR_M)
        pf_is = run_backtest(train['close'], e_is, x_is)
    except:
        pf_is = None
    
    # OOS
    try:
        e_oos, x_oos = generate_v2_signals(test['close'], test['high'], test['low'], test['volume'],
                                             atr_period=BEST_ATR_P, sma_period=BEST_SMA_P, atr_mult=BEST_ATR_M)
        pf_oos = run_backtest(test['close'], e_oos, x_oos)
    except:
        pf_oos = None
    
    m_is = extract_metrics(pf_is, f"{ticker} IS")
    m_oos = extract_metrics(pf_oos, f"{ticker} OOS")
    
    degrad = (1 - m_oos['sharpe'] / m_is['sharpe']) * 100 if m_is['sharpe'] != 0 and not np.isnan(m_oos['sharpe']) else np.nan
    
    multi_results.append({
        'ticker': ticker, **{f'is_{k}': v for k, v in m_is.items()},
        **{f'oos_{k}': v for k, v in m_oos.items()}, 'degradation': degrad
    })
    multi_portfolios[ticker] = {'is': pf_is, 'oos': pf_oos}
    
    is_sr = f"{m_is['sharpe']:.3f}" if not np.isnan(m_is['sharpe']) else "N/A"
    oos_sr = f"{m_oos['sharpe']:.3f}" if not np.isnan(m_oos['sharpe']) else "N/A"
    oos_ret = f"{m_oos['return']:.2%}" if not np.isnan(m_oos['return']) else "N/A"
    oos_dd = f"{m_oos['max_dd']:.2%}" if not np.isnan(m_oos['max_dd']) else "N/A"
    oos_wr = f"{m_oos['win_rate']:.1%}" if not np.isnan(m_oos['win_rate']) else "N/A"
    oos_pf = f"{m_oos['pf']:.2f}" if not np.isnan(m_oos['pf']) else "N/A"
    deg = f"{degrad:.1f}" if not np.isnan(degrad) else "N/A"
    
    print(f"{ticker:<8} {is_sr:>10} {oos_sr:>11} {m_is['trades']:>10} {m_oos['trades']:>11} "
          f"{oos_ret:>11} {oos_dd:>10} {oos_wr:>8} {oos_pf:>8} {deg:>8}")

# Aggregate stats
valid_oos = [r for r in multi_results if not np.isnan(r.get('oos_sharpe', np.nan)) and r['oos_trades'] > 0]
total_oos_trades = sum(r['oos_trades'] for r in valid_oos)
avg_oos_sharpe = np.mean([r['oos_sharpe'] for r in valid_oos]) if valid_oos else np.nan

print(f"\n{'='*110}")
print(f"Assets with OOS trades: {len(valid_oos)}/{len(TICKERS)}")
print(f"Total OOS trades across all assets: {total_oos_trades}")
print(f"Average OOS Sharpe: {avg_oos_sharpe:.3f}" if not np.isnan(avg_oos_sharpe) else "Average OOS Sharpe: N/A")
print(f"OOS trades/year (aggregate): {total_oos_trades / (len(valid_oos) * (1-TRAIN_RATIO) * 8):.1f}" if valid_oos else "")

## 9. Feature Ablation — What's Actually Helping?

Test each v2 improvement independently to see what's contributing:
1. Lower ATR mult alone (no other changes)
2. Trailing stop alone
3. Pullback re-entry alone
4. ADX + Volume filters alone
5. Full v2 (all combined)

In [ ]:
# Feature ablation on NVDA OOS data
print("Feature Ablation — NVDA OOS\n")
print(f"{'Variant':<35} {'Sharpe':>8} {'Return':>10} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8}")
print("-" * 80)

ablation_configs = [
    ("v1 Baseline", dict(atr_period=V1_ATR_PERIOD, sma_period=V1_SMA_PERIOD, atr_mult=V1_ATR_MULT,
                          trail_atr_mult=99, adx_min=0, vol_spike_mult=0, pullback=False)),
    ("Lower mult only (1.0)", dict(atr_period=14, sma_period=10, atr_mult=1.0,
                                    trail_atr_mult=99, adx_min=0, vol_spike_mult=0, pullback=False)),
    ("Trailing stop only", dict(atr_period=V1_ATR_PERIOD, sma_period=V1_SMA_PERIOD, atr_mult=V1_ATR_MULT,
                                 trail_atr_mult=TRAIL_ATR_MULT, adx_min=0, vol_spike_mult=0, pullback=False)),
    ("ADX + Volume only", dict(atr_period=V1_ATR_PERIOD, sma_period=V1_SMA_PERIOD, atr_mult=V1_ATR_MULT,
                                trail_atr_mult=99, adx_min=ADX_MIN, vol_spike_mult=VOL_SPIKE_MULT, pullback=False)),
    ("Pullback re-entry only", dict(atr_period=V1_ATR_PERIOD, sma_period=V1_SMA_PERIOD, atr_mult=V1_ATR_MULT,
                                     trail_atr_mult=99, adx_min=0, vol_spike_mult=0, pullback=True)),
    ("Full v2 (all features)", dict(atr_period=BEST_ATR_P, sma_period=BEST_SMA_P, atr_mult=BEST_ATR_M,
                                     trail_atr_mult=TRAIL_ATR_MULT, adx_min=ADX_MIN,
                                     vol_spike_mult=VOL_SPIKE_MULT, pullback=PULLBACK_ENABLED)),
]

ablation_results = []
for name, kwargs in ablation_configs:
    try:
        e, x = generate_v2_signals(nvda_test['close'], nvda_test['high'],
                                    nvda_test['low'], nvda_test['volume'], **kwargs)
        pf = run_backtest(nvda_test['close'], e, x)
        m = extract_metrics(pf, name)
    except:
        m = extract_metrics(None, name)
    
    ablation_results.append(m)
    sr = f"{m['sharpe']:.3f}" if not np.isnan(m['sharpe']) else "N/A"
    ret = f"{m['return']:.2%}" if not np.isnan(m['return']) else "N/A"
    dd = f"{m['max_dd']:.2%}" if not np.isnan(m['max_dd']) else "N/A"
    tr_yr = f"{m['trades_yr']:.1f}" if not np.isnan(m['trades_yr']) else "N/A"
    print(f"{name:<35} {sr:>8} {ret:>10} {dd:>8} {m['trades']:>8} {tr_yr:>8}")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
names = [r['label'] for r in ablation_results]
sharpes = [r['sharpe'] if not np.isnan(r['sharpe']) else 0 for r in ablation_results]
trades = [r['trades'] for r in ablation_results]

x = np.arange(len(names))
bars = ax.bar(x, sharpes, color=['#888'] + ['steelblue']*4 + ['darkorange'], edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('OOS Sharpe Ratio')
ax.set_title('Feature Ablation — NVDA OOS Sharpe')
ax.grid(alpha=0.3, axis='y')

# Annotate trade count on bars
for i, (bar, tr) in enumerate(zip(bars, trades)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{tr} trades', ha='center', fontsize=7, color='#555')

plt.tight_layout()
plt.show()

## 10. FTMO Monte Carlo — v2 on NVDA OOS

In [ ]:
if pf_v2_oos is not None and pf_v2_oos.trades.count() > 0:
    oos_daily_returns = pf_v2_oos.daily_returns().dropna().values
    
    n_paths = MONTE_CARLO_PATHS
    n_days = FTMO_CHALLENGE_DAYS
    account = FTMO_ACCOUNT
    target = account * FTMO_PROFIT_TARGET
    max_daily = account * FTMO_MAX_DAILY_DD
    max_total = account * FTMO_MAX_TOTAL_DD
    
    pass_count = fail_daily = fail_total = fail_time = 0
    final_pnls = []
    sample_paths = []
    
    np.random.seed(42)
    for s in range(n_paths):
        sampled = np.random.choice(oos_daily_returns, size=n_days, replace=True)
        daily_pnl = account * sampled
        cumulative_pnl = np.cumsum(daily_pnl)
        equity = account + cumulative_pnl
        
        hit_target = np.any(cumulative_pnl >= target)
        worst_daily = np.min(daily_pnl)
        max_dd_path = np.max(np.maximum.accumulate(equity) - equity)
        
        if worst_daily <= -max_daily:
            fail_daily += 1
        elif max_dd_path >= max_total:
            fail_total += 1
        elif hit_target:
            pass_count += 1
        else:
            fail_time += 1
        
        final_pnls.append(cumulative_pnl[-1])
        if s < 150:
            sample_paths.append(equity)
    
    pass_rate = pass_count / n_paths
    final_pnls = np.array(final_pnls)
    
    # Also run MC on v1 for comparison
    v1_dr = pf_v1_oos.daily_returns().dropna().values if pf_v1_oos else np.array([0])
    v1_pass = 0
    np.random.seed(42)
    for _ in range(n_paths):
        sampled = np.random.choice(v1_dr, size=n_days, replace=True)
        eq = account + np.cumsum(account * sampled)
        if np.any(eq >= account + target):
            if np.min(account * sampled) > -max_daily and np.max(np.maximum.accumulate(eq) - eq) < max_total:
                v1_pass += 1
    v1_pass_rate = v1_pass / n_paths
    
    print(f"FTMO Monte Carlo Comparison — {n_paths:,} paths x {n_days} days\n")
    print(f"{'Metric':<25} {'v1 Original':>15} {'v2 New':>15}")
    print(f"{'-'*55}")
    print(f"{'Pass Rate':<25} {v1_pass_rate:>15.1%} {pass_rate:>15.1%}")
    print(f"{'Fail (daily DD)':<25} {'—':>15} {fail_daily/n_paths:>15.1%}")
    print(f"{'Fail (total DD)':<25} {'—':>15} {fail_total/n_paths:>15.1%}")
    print(f"{'Fail (time)':<25} {'—':>15} {fail_time/n_paths:>15.1%}")
    print(f"{'Mean Final P&L':<25} {'—':>15} ${final_pnls.mean():>14,.0f}")
    print(f"{'Median Final P&L':<25} {'—':>15} ${np.median(final_pnls):>14,.0f}")
    
    improvement = (pass_rate - v1_pass_rate) / max(v1_pass_rate, 0.001) * 100
    print(f"\nPass rate change: {v1_pass_rate:.1%} -> {pass_rate:.1%} ({improvement:+.1f}%)")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'FTMO Monte Carlo — NVDA v2 | Pass Rate: {pass_rate:.1%}', fontsize=13, fontweight='bold')
    
    for path in sample_paths:
        c = 'green' if path[-1] >= account + target else ('red' if path[-1] <= account - max_total else 'gray')
        axes[0].plot(path, color=c, alpha=0.12, linewidth=0.5)
    axes[0].axhline(account + target, color='green', linestyle='--', linewidth=2, label='Target')
    axes[0].axhline(account - max_total, color='red', linestyle='--', linewidth=2, label='Max Loss')
    axes[0].set_xlabel('Trading Day'); axes[0].set_ylabel('Equity ($)')
    axes[0].set_title('Sample Equity Paths')
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    
    axes[1].hist(final_pnls, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(target, color='green', linestyle='--', linewidth=2, label=f'+${target:,.0f}')
    axes[1].axvline(-max_total, color='red', linestyle='--', linewidth=2, label=f'-${max_total:,.0f}')
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_xlabel('Final P&L ($)'); axes[1].set_ylabel('Frequency')
    axes[1].set_title('P&L Distribution')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot run FTMO Monte Carlo — no v2 OOS trades.")

## 11. Multi-Asset OOS Equity Curves

In [ ]:
# Plot OOS equity curves for all assets
n_assets = len(TICKERS)
cols = 4
rows = (n_assets + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
fig.suptitle(f'ATR Breakout v2 — OOS Equity Curves (all assets)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    pf_oos = multi_portfolios[ticker]['oos']
    
    if pf_oos is not None:
        pf_oos.value().plot(ax=ax, color='steelblue', linewidth=1.5)
        m = [r for r in multi_results if r['ticker'] == ticker][0]
        sr = m['oos_sharpe']
        tr = m['oos_trades']
        ax.set_title(f'{ticker} | Sharpe: {sr:.2f} | {tr} trades', fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'No trades', ha='center', va='center', fontsize=12, color='gray')
        ax.set_title(f'{ticker} | No trades', fontsize=9)
    
    ax.grid(alpha=0.3)
    ax.set_ylabel('$', fontsize=8)
    ax.tick_params(labelsize=7)

# Hide unused subplots
for i in range(n_assets, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

# Summary: which assets are viable?
print("\nMulti-Asset Viability Summary:")
print(f"{'Ticker':<8} {'OOS Sharpe':>10} {'Trades':>8} {'Viable?':>10}")
print("-" * 40)
for r in multi_results:
    sr = r['oos_sharpe']
    tr = r['oos_trades']
    if np.isnan(sr) or tr < 5:
        viable = "NO"
    elif sr > 0.5 and tr >= 10:
        viable = "YES"
    elif sr > 0 and tr >= 5:
        viable = "MAYBE"
    else:
        viable = "NO"
    sr_str = f"{sr:.3f}" if not np.isnan(sr) else "N/A"
    print(f"{r['ticker']:<8} {sr_str:>10} {tr:>8} {viable:>10}")

## 12. Final Assessment

In [ ]:
print("=" * 70)
print("ATR VOLATILITY BREAKOUT v2 — FINAL ASSESSMENT")
print("=" * 70)

print(f"\nBest v2 params: atr_period={BEST_ATR_P}, sma_period={BEST_SMA_P}, atr_mult={BEST_ATR_M}")
print(f"Trailing stop: {TRAIL_ATR_MULT}x ATR | ADX > {ADX_MIN} | Vol spike > {VOL_SPIKE_MULT}x")

print(f"\nTRADE COUNT IMPROVEMENT (NVDA):")
print(f"   v1 IS: {m_v1['trades']} trades ({m_v1['trades_yr']:.1f}/yr)")
print(f"   v2 IS: {int(best_v2['trades'])} trades ({best_v2['trades_yr']:.1f}/yr)")
print(f"   v2 OOS: {m_v2_oos['trades']} trades ({m_v2_oos['trades_yr']:.1f}/yr)")
ratio = int(best_v2['trades']) / max(m_v1['trades'], 1)
print(f"   Improvement: {ratio:.1f}x more trades")

print(f"\nMULTI-ASSET COVERAGE:")
viable = [r for r in multi_results if not np.isnan(r.get('oos_sharpe', np.nan)) and r['oos_trades'] >= 10 and r['oos_sharpe'] > 0.5]
marginal = [r for r in multi_results if not np.isnan(r.get('oos_sharpe', np.nan)) and r['oos_trades'] >= 5 and r.get('oos_sharpe', 0) > 0 and r not in viable]
print(f"   Viable (Sharpe > 0.5, 10+ trades): {[r['ticker'] for r in viable]}")
print(f"   Marginal (positive, 5+ trades):    {[r['ticker'] for r in marginal]}")

if 'pass_rate' in dir():
    print(f"\nFTMO VIABILITY:")
    print(f"   v1 pass rate: {v1_pass_rate:.1%}")
    print(f"   v2 pass rate: {pass_rate:.1%}")

print(f"\nHONEST WARNINGS:")
print(f"   1. v2 has more degrees of freedom than v1 (trailing stop, ADX, volume, pullback)")
print(f"      Overfit risk is higher — watch the IS/OOS degradation closely")
print(f"   2. The loop-based signal generator is slower than vectorized v1")
print(f"      Fine for daily, not suitable for intraday without optimization")
print(f"   3. Volume spike filter may miss breakouts on low-volume days")
print(f"      (legitimate gap-ups with thin volume get filtered out)")
print(f"   4. Multi-asset rotation needs a portfolio-level allocation rule")
print(f"      (equal weight? inverse vol? rank by momentum?)")
print(f"\nNEXT STEPS:")
print(f"   1. Run this on the full FTMO equity universe")
print(f"   2. Add position sizing (inverse vol targeting)")
print(f"   3. Test with 0.1% fees + 0.1% slippage for realistic friction")
print(f"   4. Combine with weak signals ensemble as signal #9")
print("=" * 70)